# Conservation & mixed method



A single boundary value problem can admit different weak formulations, possibly set in different spaces. For the Poisson equation, the *primal weak form* can be approximated by the Lagrange finite element method, while the *dual weak form* or the  *mixed weak form* can be approximated by the pair of
Raviart-Thomas and DG finite element spaces. This notebook is on the latter.


From the practical standpoint, one of the primary reasons for using this latter mixed method for the Poisson equation is **conservation**, a discrete *structure-preservation property*, which we define in this notebook.  We will use the Raviart-Thomas elements we discussed [earlier](VectorElem.ipynb) to implement and study a *conservative mixed method* in this notebook. 

## Two typical examples

Many boundary value problems arise by combining physical conservation laws with empirical constitutive laws. Methods that preserve the conservation law discretely in some sense are referred to as conservative methods.

Here are two examples of how conservation laws and constitutive laws combine to produce boundary value problems.

### Example A: Porous media flow


```{index} Darcy law
```

For slow viscous flow through a permeable porous medium, the Darcy law is an empirical statement connecting the flux of the fluid $q$ to the pressure ($p$) gradient by 




q = -K {\mathop{\mathrm{grad}}} p 


where $K$ is the permeability tensor of the rock or the porous formation. Conservation of mass then implies 


{\mathop{\mathrm{div}}} q = f 


where $f$ represents injection sources, wells, or sinks.

```{index} porous media; flow
```

```{index} porous media; permeability
```


<br>

Historically, Example A was studied extensively by the fossil fuel industry. However, it also finds ecological uses in groundwater remediation. And, perhaps even more fittingly for us here in Portland, in modeling the percolation of hot water through [coffee grounds](https://en.wikipedia.org/wiki/Darcy%27s_law#Use_in_coffee_brewing)!

```{index} coffee grounds
```



### Example B: Diffusion of heat


```{index} heat flux; Fourier's law
```

Consider the stationary heat dissipation example from [a prior notebook](Confusion.ipynb). Recall that Fourier's law of heat conduction is a constitutive law that states (in the absence of advective and radiative effects) that the flux of heat energy $q$ (sometimes also called heat flux density) through a material is related to the gradient of the temperature $T$ by

 

q = -\kappa \,{\mathop{\mathrm{grad}}} T


where $\kappa$ is the measured thermal conductivity of the material. Conservation of energy implies that


    {\mathop{\mathrm{div}}} q = f 


where $f$ represents the heat source. 


Note that in both examples, elimination of flux results in an equation of the form  




 

-{\mathop{\mathrm{div}}} (\alpha {\mathop{\mathrm{grad}}} u) = f


(with $u$ equal to either $T$ or $p$, and $\alpha$ equal to either $\kappa$ or $K$), which we have treated previously using Lagrange elements. In this notebook, rather than eliminating $q$, our focus is on approximating $q$.  

## Conservation in the finite element context


Suppose we use a mesh $


 

{\varOmega_h}$ of a $d$-dimensional domain ${\varOmega}$ to compute a discrete approximation $q_h$ to the exact flux $q$. Let  $P_p({\varOmega_h}) = \prod_{K \in {\varOmega_h}} P_p(K)$ denote the DG space, so piecewise polynomial vector fields of degree at most $p$ are in $P_p({\varOmega_h})^d.$

**Definition:** We say that a vector field $q_h$ in $P_p({\varOmega_h})^d$ is a **conservative flux** approximation of an exact flux $q$ if the following two conditions hold:

- **Condition (1):** on all interior mesh facets, &LeftDoubleBracket; $q_h\cdot n$ &RightDoubleBracket; $ = 0$, and 
- **Condition (2):** on every mesh element $K \in {\varOmega_h}$, 


\int_{\partial K}  q_h \cdot n = \int_{\partial K}  q\cdot n.


```{index} conservative flux; single-valued flux 
````

```{index} conservative flux; element-wise conservation 
````

```{index} conservative flux; definition  
````

Condition (1) requires the normal flux to be single-valued. There we have used the notation &LeftDoubleBracket; $q_h\cdot n$  &RightDoubleBracket; for the jump of the normal component of the flux $q$. 

Condition (2) is an element-wise conservation condition. Note that by the divergence theorem the right hand side of the equality in Condition (2) also equals 




 

\int_{\partial K} q\cdot n = \int_K {\mathop{\mathrm{div}}} q = \int_K f


with $f = {\mathop{\mathrm{div}}} q.$

Also note that both Conditions (1) and (2) refer only to the values of $q_h$ on the facets. (Indeed, there are methods that produce flux approximations only along the mesh facets, and the same definition can be used to decide if such fluxes are conservative or not. The mixed method however will produce fluxes on the whole domain.)

```{index} control volume; conservation law
```

To understand why  Conditions (1) and (2)  are interesting properties for an approximation to have, consider one of the above-mentioned applications, say the diffusion of heat, and recall how the conservation equation arises. In physics,  conservation of energy is often written in *integral form* over a control volume $S$ as 


\int_{\partial S} q \cdot n = \int_S f,


i.e., the flux of heat leaving the subdomain $S$ through its boundary (equalling the left hand side above) must equal the amount of heat produced by sources $f$ within $S$ (equal to the right hand side above). Because this conservation statement in integral form  holds for *any* control volume $S$, the divergence theorem implies the equivalent differential equation 
$


 

{\mathop{\mathrm{div}}} q = f$, which is one of the conservation equations stated previously. 

```{index} conservation law; integral form 
```


```{index} control volume; discrete
```

Now, consider how we may obtain a *discrete version of the integral form* of the conservation law. Instead of arbitrary control volumes, we now restrict ourselves to **discrete control volumes,** which are the unions of the  elements used in the computation. Namely, consider  any subset of mesh elements $


 

O_h \subseteq {\varOmega_h}$ and put  $S_h = \cup\{ K: K \in O_h\}$. If the discrete flux out of the subdomain $S_h$ satisfies 

\tag{DC}
\int_{\partial S_h}  q_h \cdot n = \int_{S_h} f,


then it makes sense to declare that $q_h$ satisfies a discrete conservation law, since equation (DC) is exactly the  discrete analog of the exact integral form of the conservation law, namely the equation 
$
\int_{\partial S} q \cdot n = \int_S f
$
discussed previously. The only difference is that while the exact form holds on any $S$, equation (DC) only holds for the discrete control volumes $S_h$.

One can immediately verify that the above equation (DC) is a direct consequence of Conditions (1) and (2) in our definition. Note how we need **both** Conditions (1) and (2) to accomplish the interelement cancellations of influx and outflux within $S_h$.

One of the modern themes in numerical solution of partial differential equations, called  **structure preservation**, studies questions like this: how can we construct methods that (not only converge optimally, but also) yield solutions that preserve certain critical features or structures of the exact solution? In the context of our model problem, methods that produce a conservative flux $q_h$, as defined above, are structure-preserving, the preserved structure being a conservation law.

```{index} structure preservation
```

```{index} superconvergence; conservation
```

Another way of thinking about conservation is in terms of **superconvergence** of certain functionals. (The phenomena where the error, or some functional of the error, goes to zero at an unexpectedly high speed is called superconvergence. We have seen a superconvergence example in [a prior notebook](VectorElem.ipynb) in a completely different context.)  Condition (2) in the definition of conservation, 
$
\int_{\partial K} q_h \cdot n  = \int_{\partial K} q\cdot n, 
$
can equivalently  be written using  the functional 

$$G_K (r) = \int_{\partial K} r \cdot n,
$$

as 

$$
G_K(q) = G_K (q_h).
$$

For good methods,  we expect the error $q_h - q$ to go zero as the mesh size $h \to 0$, so  we expect $G_K(q) - G_K(q_h) \to 0$. But it is exceptional to get zero for a functional of the error. For a conservative method, that exceptional property holds and $G_K(q) - G_K(q_h) = 0$.

## A simple model problem

Let us solve a  specific example  built from Example B. We want to simulate a steady heat flux  $


 

q: {\varOmega} \to {\mathbb{R}}^2$ on a rectangular bar-shaped domain ${\varOmega}$ of length 6 units and width 2 units, divided into equal left and right halves. More specifics of the problem appear after we draw the geometry.

In [1]:
import ngsolve as ng
from ngsolve.webgui import Draw
from netgen.occ import X, Y, MoveTo, Rectangle, Glue, OCCGeometry

rgtbar = Rectangle(3, 2).Face()
lftbar = MoveTo(-3, 0).Rectangle(3, 2).Face()
lftbar.faces.name = 'lftbar'
rgtbar.faces.name = 'rgtbar'
lftbar.edges.Min(X).name = 'lft'
lftbar.edges.Min(Y).name = 'bot'
lftbar.edges.Max(Y).name = 'top'
lftbar.edges.Max(X).name = 'mid'
rgtbar.edges.Max(X).name = 'rgt'
rgtbar.edges.Min(Y).name = 'bot'
rgtbar.edges.Max(Y).name = 'top'
bar = Glue([lftbar, rgtbar])
Draw(bar);
geo = OCCGeometry(bar, dim=2)
mesh = ng.Mesh(geo.GenerateMesh(maxh=1/4))

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'ngsolve_version': 'Netgen x.x', 'mesh_dim': 3…

Here are the further specifications of the thermal problem:

```{index} thermal conductivity
```

- Materials whose isotropic *thermal conductivity* ($\kappa$) values equal 1 and 10 occupy the   left and right halves, respectively.  
- The left boundary edge of  $\Omega$ is *kept at temperature* $T=1$ while the right edge is kept  at $T=1/10$. 
- The top and bottom boundary edges are *perfectly insulated,*  i.e, the outward flux $q \cdot n$ vanishes there. 
- The bar is heated by a source which is modeled by 
$$
  f(x, y) = 5 \exp(-10 [(x/5)^2 + (y-1)^2]).
$$  


```{index} heated plate
```

```{index} insulator
```


Here is a plot of $f$:

In [2]:
from ngsolve import x, y, exp
f = 5 * exp(-10*( (x/5)**2 + (y-1)**2))
Draw(f, mesh);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

Clearly, this heat source appears centered in the domain. However, the material is not homogenous as seen from the thermal conductivity plot next:

In [3]:
kappa = mesh.MaterialCF({'lftbar': 1, 'rgtbar': 10})
Draw(kappa, mesh);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

Finally, the other piece of given information on boundary conditions tells us that heat may enter the bar through the left end. Here is a visualization of the boundary data extension:

In [4]:
Tbdr = mesh.BoundaryCF({'lft': 1, 'rgt': 0.1})
Tboundary = ng.GridFunction(ng.H1(mesh)); Tboundary.Set(Tbdr, definedon=mesh.Boundaries('lft|rgt'));
Draw(Tboundary);

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {}, 'ngsolve_version': '6.2.26…

Given all the above bits of information, the problem is to compute the temperature $T$ and the heat flux $q$.  We will solve it momentarily. 

**Questions for discussion:**

- Do you have any physical guesses on which side will get heated up more in steady state?
- What features in the problem generates heat and which ones dissipate heat?
- Will the centered source create a steady hot spot in the center?
  

## The Raviart-Thomas mixed method

Consider the boundary value problem of finding 
$


 

u: {\varOmega}\to {\mathbb{R}}$ and $q: {\varOmega} \to {\mathbb{R}}^d$ ($d\ge 2$), given $f: {\varOmega} \to {\mathbb{R}}$ and (pointwise) invertible $\kappa : {\varOmega} \to {\mathbb{R}}^{d \times d}$, satisfying 


\kappa^{-1} q + {\mathop{\mathrm{grad}}} u =0, \qquad {\mathop{\mathrm{div}}} q = f, 


with Dirichlet boundary conditions $u =0 $ on $\partial {\varOmega}$. From the lectures, we know that the **mixed weak formulation** of this problem reads as follows.

<font color=blue>

>Find $


 

u \in L^2({\varOmega})$ and $q \in H({\mathop{\mathrm{div}}}, {\varOmega})$ satisfying 
>
>
\begin{aligned}
\int_{\varOmega} \kappa^{-1} q \cdot r - \int_{\varOmega} u\, {\mathop{\mathrm{div}}} r & = 0, 
&& \text{ for all } r \in H({\mathop{\mathrm{div}}}, {\varOmega}), \text{ and }
\\
\int_{\varOmega} v\, {\mathop{\mathrm{div}}} q & = \int_{\varOmega} f \,v,  
&& \text{ for all } v \in L^2({\varOmega}).
\end{aligned}

>
</font>


To obtain a numerical method, starting from the above weak form, 
we use the  DG space $P_p({\varOmega_h})$ in place of $L^2({\varOmega})$ and the Raviart-Thomas finite element space (introduced [earlier](VectorElem.ipynb)) 




 

R_{h, p} =
\left\{
q\in H({\mathop{\mathrm{div}}}, {\varOmega}) : \text{ for all } K \in {\varOmega_h}, \quad  q|_K \in P_p(K)^2 + \begin{pmatrix} x \\ y \end{pmatrix} P_p(K) \; \right\}


in place of $H({\mathop{\mathrm{div}}}, {\varOmega}).$ 

<font color=blue>
    
> The **Raviart-Thomas (RT) mixed method** of order $p$ finds $u_h \in  P_p({\varOmega_h})$ and
> $q_h \in R_{h, p}$ satisfying 
>
>


 

\begin{aligned}
\int_{\varOmega} \kappa^{-1} q_h \cdot r_h - \int_{\varOmega} u_h\, {\mathop{\mathrm{div}}} r_h & = 0, 
&& \text{ for all } r_h \in R_{h, p}, \text{ and }
\\
\int_{\varOmega} v_h\, {\mathop{\mathrm{div}}} q_h & = \int_{\varOmega} f \,v_h,  
&& \text{ for all } v_h \in P_p({\varOmega_h}).
\end{aligned}

>

</font>

```{index} mixed method; Raviart-Thomas
```

In the latter equation, if we select a test function 
$


 

v_h$ that is the indicator function of one element $K \in {\varOmega_h}$, which is of course contained in $P_p({\varOmega_h})$ for any $p \ge 0$, then we obtain 


\int_K {\mathop{\mathrm{div}}} q_h = \int_K f, 


which is equivalent to Condition (2) in our definition of a conservative flux. Of course, Condition (1) is automatically satisfied by $q_h$, since every $q_h \in R_{h, p}$ has 
&LeftDoubleBracket; $q_h\cdot n$ &RightDoubleBracket; $ = 0$. This proves that  **the Raviart-Thomas mixed method yields conservative fluxes.** 

Next, we proceed to implement this method in NGSolve. There are two ways to assemble such a mixed weak form, as we shall see in the next section. 

## Two ways to assemble
 

 
```{index} mixed method; assemble big bilinear form
```
#### The first way: assemble a composite form

We can  think of the system using a *"big" bilinear form* $C(\cdot, \cdot)$ in the product space $



 

R_{h, p} \times P_p({\varOmega_h})$, i.e., for any  $(q_h, u_h), (r_h, v_h) \in 
R_{h, p} \times P_p({\varOmega_h})$, set 


C((q_h, u_h), (r_h, v_h)) = 
\int_\Omega \kappa^{-1} q_h \cdot r_h - \int_\Omega u_h \,\text{div } r_h
- \int_\Omega v_h \,\text{div } q_h.


Then the previous set of two equations for the Raviart-Thomas mixed method can be described as the single equation using the composite bilinear form:


C((q_h, u_h), (r_h, v_h)) = -\int_{\varOmega} f v_h,
\qquad \text{ for all } v_h \in P_p({\varOmega_h}), \, r_h \in R_{h,p}.


To compute with this form, we need the product space 
$


 

R_{h, p} \times P_p({\varOmega_h})$. This space is represented by the object `RW` introduced below. 

In [5]:
p = 4 
R = ng.HDiv(mesh, order=0, RT=True)
W = ng.L2(mesh, order=0)
RW = R * W   # Alternately:  RW = ng.FESpace([R, W])

Note that trial and test functions in `RW` have two components. Otherwise the procedure for declaring  the bilinear form is exactly the same as what we saw previously. 

In [6]:
from ngsolve import div, dx
q, u = RW.TrialFunction() 
r, v = RW.TestFunction()
C = ng.BilinearForm(RW)
C += ((1/kappa) * q*r - u*div(r) - div(q)*v) * dx
C.Assemble(); type(C.mat)

ngsolve.la.SparseMatrixd

We now have a sparse assembled matrix representing the  matrix of the big bilinear form $C(\cdot, \cdot)$ on the product finite element space.

#### The second way: assemble a block system

```{index} mixed method; assemble blocks
```

In the second approach, we make individual components of the bilinear form and put them together, i.e.,  we make the following *"small" bilinear forms*:

$$
a(q, r) = \int_\Omega \kappa^{-1} q \cdot r, \qquad
b(q, v) = -\int_\Omega v \,\text{div } q
$$

Then the equations RT mixed method can be rewritten using these forms as 

$$
\begin{aligned}
a(q_h, r_h) + b(r_h, u_h) & = 0 
&& \text{ for all } r_h \in R_{h, p}, \text{ and }
\\
\,b( q_h, v_h)  & = -\int_{\varOmega} f \,v_h,  
&& \text{ for all } v_h \in P_p({\varOmega_h}).
\end{aligned}
$$

Note that the bilinear form $b$  uses different trial and test space. Implementing this requires the use of keyword arguments `trialspace` and `testspace` that we have not seen previously, but is self explanatory. There is no need to define a product space in this approach. 

In [7]:
q, r = R.TnT()
u, v = W.TnT()

b = ng.BilinearForm(trialspace=R, testspace=W)
b += -div(q)*v * dx

a = ng.BilinearForm(R)
a += (1/kappa) * q*r * dx

b.Assemble(); a.Assemble();

When working with such component forms, you also need a facility to place them into appropriate places of a block partitioned matrix. Let $\mathtt A$ and $\mathtt B$  denote the stiffness matrices of the forms $a(\cdot, \cdot)$ and $b(\cdot, \cdot)$ made in the code block above, respectively. Then, what  we want is the matrix 

$$
\mathbb B = 
\begin{bmatrix}
\mathtt A & \mathtt B^T \\ 
\mathtt B & 0 
\end{bmatrix}
$$

where ${}^T$ denotes the transpose. 

In [8]:
BB = ng.BlockMatrix([[a.mat, b.mat.T], 
                     [b.mat, None   ]])
type(BB)

ngsolve.la.BlockMatrix

This `BB` object represents the above-mentioned block matrix $\mathbb B$.
Note that the `None` element in the code above is how we represent  the zero block sparsely.

```{index} block matrix; in mixed systems
```

Obviously $\mathbb B$ and $C$ represent the same linear operator. Block structures are just a convenience feature to work with components (without replicating the already allocated memory of the components).  Block matrices expect block vectors when asked to perform matrix multiplication. 

Be warned that it's not a good idea to directly perform a generic vector operation between a block vector object and a regular NGSolve vector. In some cases NGSolve will raise an exception and will not allow you to do so. Also, when you work with block vectors, it is a good idea to ensure that you have variable names bound to its individual component vectors in your python workspace.  Here is an exercise to get some practice on working with the `BlockMatrix` structures and with matrix-vector products in NGSolve. 

```{index} block matrix; in ngsolve
```

**Question for discussion:**

- How would you write a python code to verify that the application of the block matrix $\mathbb{B}$  and the matrix $C$ of the composite bilinear form to the same vector gives the same result. (*Suggestion:*  Look up the `component` attribute of an NGSolve grid function on a product space.)

<b><font color=red>Solution:</font></b>

In [9]:
def ApplyBB(q, u, R, W):
    """ Return tuple of components of BB * [q; u] 
    assuming BB is made in enclosing scope. """
    
    quvec = ng.BlockVector([q, u])
    
    # Allocate space to store the matrix-vector product
    BBq = q.CreateVector()
    BBu = u.CreateVector()
    BBqu = ng.BlockVector([BBq, BBu])

    # Compute the product as a BlockVector
    BBqu.data = BB * quvec
    
    return BBq, BBu


def ApplyBigC(q, u, RW):
    """
    Return BigB * [q; u] as a product space (RW) grid function, 
    assuming C is the BigB matrix made in the enclosing scope."""
    
    # Allocate space for the result in a product FESpace
    quh = ng.GridFunction(RW)
    quh.components[0].vec.data = q
    quh.components[1].vec.data = u

    # Matrix-vector multiply in a product FESpace GridFunction
    BigCquh = ng.GridFunction(RW)
    BigCquh.vec.data = C.mat * quh.vec
    
    return BigCquh

def BlockMinusLong(qblock, ublock, qulonggf):
    """
    Given inputs qulonggf (a product space gridfunction)  and 
    qublock= [qblock, ublock] (block vector components), return
    qulong after doing   qulong -= qublock """

    qulonggf.components[0].vec.data -= qblock
    qulonggf.components[1].vec.data -= ublock
    return qulonggf

In [10]:
qh = ng.GridFunction(R)
uh = ng.GridFunction(W)
qh.Set((x, y))
uh.Set(x*y)
    
BBqh, BBuh = ApplyBB(qh.vec, uh.vec, R, W)
BigCquh = ApplyBigC(qh.vec, uh.vec, RW) 

diff = BlockMinusLong(BBqh, BBuh, BigCquh)
print('The difference between the two matrix actions on the same vector is\n', 
      diff.vec.Norm())

The difference between the two matrix actions on the same vector is
 6.188405156894679e-15


## Solving for the thermal flux 

We now return to solve model problem stated earlier. To repeat the problem, translating it to a boundary value problem, we need to solve for temperature $T$ satisfying 



\begin{aligned}
\kappa^{-1} q + {\mathop{\mathrm{grad}}} T & = 0, && \text{ in } {\varOmega}
\\
{\mathop{\mathrm{div}}} q & = 5\, e^{-10 [(x/5)^2 + (y-1)^2]},  && \text{ in } {\varOmega}
\\
q\cdot n & = 0 && \text{ on top and bottom boundaries} 
\\
T & = 1  && \text{ on the left boundary } {\partial{\varOmega}_{\text{left}}} 
\\
T & = 1/10 && \text{ on the right boundary } {\partial{\varOmega}_{\text{right}}}.
\end{aligned}


To solve this using the mixed method, we have to carefully incorporate the boundary conditions into the mixed formulation. Dirichlet and Neumann conditions are respectively imposed as natural and essential conditions *in the mixed formulation, in  exactly the opposite* manner of the primal formulation. We use the subspace of the Raviart-Thomas space where the flux boundary conditions are incorporated essentially:



{\mathring{R}_{h, p}^{\text{top, bot}}}= \{ r \in R_{h, p}: r\cdot n = 0 \text{ on the top and bottom boundaries}\}.


Also incorporating the Dirichlet boundary conditions on $T$ naturally, we obtain the following equations of the method: find $T_h \in P_p({\varOmega_h})$ and $q_h \in {\mathring{R}_{h, p}^{\text{top, bot}}}$ satisfying 


\begin{aligned}
\int_{\varOmega} \kappa^{-1} q_h \cdot r_h - \int_{\varOmega} T_h\, {\mathop{\mathrm{div}}} r_h & = -\int_{{\partial\varOmega_{\text{left}}}} 10\, r_h \cdot n, 
&& \text{ for all } r_h \in {\mathring{R}_{h, p}^{\text{top, bot}}} \text{ and }
\\
\int_{\varOmega} v_h\, {\mathop{\mathrm{div}}} q_h & = \int_{\varOmega} f \,v_h,  
&& \text{ for all } v_h \in P_p({\varOmega_h}).
\end{aligned}


#### Block linear system 

Adopting  the second approach to assembly mentioned above, we produce a block linear system, whose right and left hand sides are returned by the next function. Also note that to make the right hand side of the first equation, we  need to integrate  the trace of $r_h \cdot n$ along boundary edges, which is accomplished below by `r.Trace() * n`  and `ds` provided by NGSolve.

```{index} Trace; Raviart-Thomas space
```

In [11]:
from ngsolve import dx, ds

def MakeMixed(mesh, f, Tbdr, kappa, p=4):    
    """Return the block linear system (rhs and lhs) of the
    RT mixed method for solving Example T"""
    
    R = ng.HDiv(mesh, order=p, RT=True, dirichlet='top|bot')
    W = ng.L2(mesh, order=p)
    q, r = R.TnT()
    u, v = W.TnT()
    dl = ds(definedon=mesh.Boundaries('lft|rgt'))
    n = ng.specialcf.normal(mesh.dim)

    b = ng.BilinearForm(trialspace=R, testspace=W)
    b += -div(q) * v * dx
    a = ng.BilinearForm(R)
    a += (1/kappa) * q * r * dx    
    fR = ng.LinearForm(R)
    fR += -Tbdr * (r.Trace() * n) * dl
    fW = ng.LinearForm(W)
    fW += -f * v * dx
    with ng.TaskManager():
        b.Assemble() 
        a.Assemble()
        fR.Assemble()
        fW.Assemble() 
    BB = ng.BlockMatrix([[a.mat, b.mat.T], [b.mat, None]])
    FF = ng.BlockVector([fR.vec, fW.vec])   
    return BB, FF, R, W

In [12]:
BB, FF, R, W = MakeMixed(mesh, f, Tbdr, kappa)

Next, we need to solve for a vector `x` satisfying the block system `BB * x = FF`.  In previous studies, we assembled sparse matrix objects which we could hand over to a sparse solver (using the `Inverse(...)` method). However, the block matrix `BB` does not have a sparse  `Inverse` method.  Hence let us take a moment to consider an iterative solver. 

```{index} block matrix; solver
```


#### Iterative solver

Iterative solvers built using Krylov spaces do not generally require an assembled matrix, only an object that can perform the associated matrix-vector multiplication. The block matrix `BB` can perform the multiplication `BB * x`.  Although Krylov space iterative solvers can be easily implemented in a few lines of python code, let us make use of a readymade Krylov solver implementation that NGSolve provides. **MINRES** is an iterative method suitable for invertible symmetric (not necessarily positive definite) linear systems, like the block system we have produced. Its implementation is available in `ng.solvers.MinRes`.

```{index} MinRes
```

```{index} iterative solver
```


The rate of convergence of iterative solvers like MINRES depends on the condition number of the system. Hence they are usually effective only when we also provide it a good preconditioner. Recall that a **preconditioner** for use in solving $\mathbb B x = b$ is a linear operator $\mathbb P$ (acting on the same-size vectors as $\mathbb B$) with the property that $\mathbb P \mathbb B$ is better conditioned than $\mathbb B$.  Once a preconditioner is given, the iterative solver, instead of solving $\mathbb B x = b$, solves the equivalent but better conditioned system $\mathbb P\mathbb B x = \mathbb P b$ behind the scenes.

```{index} preconditioner
```


Of course, one could  set $\mathbb P$ to the identity operator on the free degrees of freedom, just to witness the performance of solver without preconditioning. (No preconditioning is the same as setting preconditioner to identitity.) Here is how we set the needed block identity operator.

```{index} double: identity; projector
```

```{index} projector; mask
```

```{index} identity; preconditioner
```

```{index} block vector
```

In [13]:
identityR = ng.Projector(mask=R.FreeDofs(), range=True)  # project to range=true/false bits of mask
identityW = ng.IdentityMatrix(W.ndof)
PI = ng.BlockMatrix([[identityR, None],
                     [None, identityW]])

In [14]:
qT = ng.BlockVector([ng.GridFunction(R).vec, 
                     ng.GridFunction(W).vec])
ng.solvers.MinRes(mat=BB, pre=PI, rhs=FF, sol=qT, maxsteps=30);

LinearSolver iteration 1, residual = 2.9349850200882894     
LinearSolver iteration 2, residual = 2.8525538570019844     
LinearSolver iteration 3, residual = 2.294830356474514     
LinearSolver iteration 4, residual = 2.2023632468343575     
LinearSolver iteration 5, residual = 1.7265143891534573     
LinearSolver iteration 6, residual = 1.591797669476658     
LinearSolver iteration 7, residual = 1.406188933386344     
LinearSolver iteration 8, residual = 1.3282590813510842     
LinearSolver iteration 9, residual = 1.179691970585117     
LinearSolver iteration 10, residual = 1.1020337549793986     
LinearSolver iteration 11, residual = 0.995812038645757     
LinearSolver iteration 12, residual = 0.9656993370434038     
LinearSolver iteration 13, residual = 0.8865937134575593     
LinearSolver iteration 14, residual = 0.8685093304602324     
LinearSolver iteration 15, residual = 0.7996103532005372     
LinearSolver iteration 16, residual = 0.7911405926295936     
LinearSolver iteration

<br>

As you can see the convergence is abysmally slow. 

**Question for discussion:**

- Can you get these iterations to reduce the error further? (Increase `maxsteps` and run again to see how many iterations you might need to do before getting anywhere close to convergence.)

We need a better preconditioner.  For our block matrix 

$$
\mathbb B = 
\begin{bmatrix}
\mathtt A & \mathtt B^T \\ 
\mathtt B & 0 
\end{bmatrix},
$$

one technique is to construct a preconditioner in the following form, using the same block partitioning,

$$
\mathbb P = 
\begin{bmatrix}
\mathtt M_R^{-1} & 0 \\
0  & \mathtt M_W^{-1}
\end{bmatrix},
$$

where $\mathtt M_R$ and $\mathtt M_W$ are the stiffness matrices of the bilinear forms 
$
\int_{\varOmega} \kappa^{-1} q\cdot r + \int_{\varOmega} ({\mathop{\mathrm{div}}} q ) ({\mathop{\mathrm{div}}} r),$
for $q, r \in {\mathring{R}_{h, p}^{\text{top, bot}}}$
and 
$
\int_{\varOmega} u \, v,$
for $u, v \in P_p({\varOmega_h}),$
respectively. Since $\mathtt M_W$ is block diagonal with one block for each element (why?), it is very easy to invert. So the cost of construction of this preconditioner is essentially the cost of inversion of $\mathtt M_R$,  a smaller matrix than the original $\mathbb B$.

```{index} preconditioner; block preconditioner for mixed systems
```


In [15]:
def MakePrec(R, W):
    q, r = R.TnT()
    mR = ng.BilinearForm(((1/kappa)*q*r + div(q)*div(r))*dx)
    mR.Assemble()
    P = ng.BlockMatrix([[mR.mat.Inverse(R.FreeDofs()), None],
                        [None,          W.Mass(1).Inverse()]])
    return P

In [16]:
PP = MakePrec(R, W)

In [17]:
qh = ng.GridFunction(R)
Th = ng.GridFunction(W)
qT = ng.BlockVector([qh.vec, Th.vec])
ng.solvers.MinRes(mat=BB, pre=PP, rhs=FF, sol=qT, maxsteps=15);

LinearSolver iteration 1, residual = 4.655166599019508     
LinearSolver iteration 2, residual = 4.650238209773678     
LinearSolver iteration 3, residual = 1.6631181609423022     
LinearSolver iteration 4, residual = 0.3060735234476721     
LinearSolver iteration 5, residual = 0.03971097203811199     
LinearSolver iteration 6, residual = 0.0020292060009608147     
LinearSolver iteration 7, residual = 0.00012449831832854423     
LinearSolver iteration 8, residual = 3.397171011593384e-06     
LinearSolver iteration 9, residual = 7.718494250916117e-08     


<br>

**Question for discussion:**

- Why does this preconditioner work well?
  

#### Temperature and flux 


`MinRes` overwrites the solution into the same memory location as the input `qT`. It is in the form of a block vector, whose components occupy the same memory as `qh` and `Th`. So to plot the temperature, we can directly use the variable name `Th` which is bound to the memory block of the $T$-component.

In [18]:
Draw(Th, deformation=True, settings={"Colormap":{"autoscale": True, "ncolors": 20}, "camera":{"transformations" :[{"type": "rotateX", "angle": -50}, {"type": "rotateY", "angle": -20}, {"type": "rotateZ", "angle": -10}]}});

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'Colormap': {'autoscale': Tru…

This plot of the calculated temperature $T$ reveals lower temperature variations on the right than on the left. You also see that there is a kink at the middle interface (indicating a discontinuity in ${\mathop{\mathrm{grad}}} T$) to accomodate a more flat $T$ profile on the right.

Next, we plot the heat flux vector field:

In [19]:
Draw(qh, vectors={'grid_size': 30}, settings={"Colormap":{"autoscale": True, "ncolors": 20}, "Light": {"ambient":0.1, "diffuse":0.7, "shininess": 10,"specularity": 0.9}});    

WebGuiWidget(layout=Layout(height='50vh', width='100%'), value={'gui_settings': {'Colormap': {'autoscale': Tru…

It is clear from from this plot that the heat flow on the right subdomain is more than on the left (as the colors represent the magnitude of the heat flux). In fact, from the direction of the fluxes, we see that the left boundary condition is helping to dissipate some of the heat generated by the source. Also observe the apparent continuity of the normal component of the heat flux despite the jump in the material properties. This is exactly as it should be for a conservative method, as otherwise nonphysical energy creation or energy loss will be predicted across the interface.

Let us also note that even though the blob of heat source was centered in the domain, it created an uncentered temperature peak to the left.  Is this shift of the peak due to the difference in boundary conditions on the left and right, or due to the difference in the left and right thermal conductivity?   The utility of having a simulation tool is that we can now answer questions like this easily even when the answers are not intuitive. Go ahead and experiment, changing the boundary conditions, and changing the thermal conductivites set in the code cells above as you please.  You will then be led to the conclusion that the shift of the peak is primarily due to the fact that the right half of the domain, due to its higher thermal conductivity, can transmit heat through it more efficiently, flattening out any temperature peaks there. The plot of  the computed thermal fluxes further confirms this by showing high fluxes in the right half.

**Questions for discussion,** *on comparison with the primal formulation:*

- How will you solve the same model problem using Lagrange finite elements applied to the primal weak form? In such an approach you will compute an approximate temperature, which we denote by  $\tilde T_h$.

- Using your computed temperature $\tilde T_h$ (in the Lagrange space $ V_{h, p}$) in your approach above, how will you compute an element-wise approximate flux $\tilde{q}_h = -\kappa {\mathop{\mathrm{grad}}} \tilde{T}_h$?  Can this flux be conservative?

- Can you quantify the spurious heat energy loss across $\Gamma$ in such a method?   Specifically, letting $\Gamma$ denote the middle interface, how will you compute 
$\int_\Gamma$  &LeftDoubleBracket; $\tilde q_h\cdot n$ &RightDoubleBracket; ? (*Suggestion:* To integrate multivalued functions on element interfaces, `dx(element_boundary=True)` can be useful.)

```{index} flux; non-conservative flux
```

```{index} double: flux jump; energy lost
```

```{index} element boundary integrals; to compute nonconservative flux jump
```

```{index} jump; of normal flux 
```

 <b><font color=red>Solution:</font></b>  

In [20]:
from ngsolve import grad, dx, ds

def SolvePrimal(mesh, f, Tbdr, kappa, p=2):
    V = ng.H1(mesh, order=p, dirichlet='lft|rgt')
    u, v = V.TnT()
    a = ng.BilinearForm(V)
    a += kappa * grad(u) * grad(v) * dx 
    a.Assemble()
    b = ng.LinearForm(V)
    b += f * v * dx
    b.Assemble()

    T = ng.GridFunction(V)
    T.Set(Tbdr, ng.BND)
    b.vec.data -= a.mat * T.vec
    T.vec.data += a.mat.Inverse(V.FreeDofs()) * b.vec
    return T

In [21]:
Th = SolvePrimal(mesh, f, Tbdr, kappa, p=0)
qh2 = -kappa * grad(Th)

In [22]:
mesh.GetBoundaries()

('bot', 'mid', 'top', 'lft', 'bot', 'rgt', 'top')

In [23]:
n = ng.specialcf.normal(2)
F = ng.FacetFESpace(mesh, order=0)
indicator = ng.GridFunction(F)  # indicator function of mid line
indicator.Set(1, definedon=mesh.Boundaries('mid'))

In [24]:
# Check that the integral of indicator at least gives the correct length 
ng.Integrate(indicator*dx(element_boundary=True), mesh)

3.9999999999999876

We compute $\int_\Gamma$  &LeftDoubleBracket; $\tilde q_h\cdot n$ &RightDoubleBracket;  for the Lagrange method's fluxes:

In [25]:
n = ng.specialcf.normal(2)
ng.Integrate(indicator*qh2 * n * dx(element_boundary=True), mesh)

-0.6133335628434815

We compute $\int_\Gamma$  &LeftDoubleBracket; $ q_h\cdot n$ &RightDoubleBracket;  for the previously computed mixed method flux $q_h$: 

In [26]:
n = ng.specialcf.normal(2)
ng.Integrate(indicator* qh * n * dx(element_boundary=True), mesh)

-5.551115123125783e-16

**Question for discussion,**  *on use of other space in the mixed formulation:*

- What happens if, in the mixed formulation, you replace the RT space ${\mathring{R}_{h, p}^{\text{top, bot}}}$ with its subspace of Cartesian product of Lagrange spaces of degree at most $p$, say for $p=1$?
  

<b><font color=red>Solution:</font></b>  

In [27]:
p=1
X = ng.H1(mesh, order=p)
Y = ng.H1(mesh, order=p, dirichlet='top|bot')
W = ng.L2(mesh, order=p)
X.ndof + Y.ndof, W.ndof

(502, 1308)

From the `ndof` counts above it looks like the possibility of $B: V \to W$ being a surjection is *not ruled out.* Recall that surjectivity of $B$, or in finite dimensional spaces, the equivalent injectivity of $B^t,$ is needed for the invertibility of 
$$
\begin{bmatrix}
A & B^t \\
B & 0 
\end{bmatrix}.
$$

In [28]:
VW = ng.FESpace([X, Y, W])
qx, qy, u = VW.TrialFunction() 
rx, ry, v = VW.TestFunction()
divq = grad(qx)[0] + grad(qy)[1]
divr = grad(rx)[0] + grad(ry)[1]
q = ng.CF((qx, qy))
r = ng.CF((rx, ry))
C = ng.BilinearForm(VW)
C += (q*r - u*divr - divq*v) * dx
C.Assemble();

*Demostration of failure to invert:*

In [29]:
try:
    C.mat.Inverse(VW.FreeDofs())
except:
    print('***COULD NOT SOLVE! Inverse raised exception***')

***COULD NOT SOLVE! Inverse raised exception***


## Summary

- We studied conservation as a discrete structure-preservation property.
- Conservative fluxes were defined in the finite element context.
- Conservation was seen as a superconvergence property. 
- The Raviart-Thomas mixed method produces conservative fluxes by design.
- Two ways to assemble the linear system of a mixed method were seen: one  way is to make a composite bilinear form and the other is to make component bilinear forms and put them together in a block matrix.
- Iterative solvers like MINRES can be used to solve the block mixed systems.
- A block preconditioner using inverses of the stiffness matrices of the component bilinear forms was effective.
- Conservative fluxes do not produce spurious energy loss across interfaces.
- The choice of spaces in the mixed formulation is crucial.